# WAI NSFW v15 — Colab Studio ⚡

โน้ตบุ๊กสำหรับดาวน์โหลดและใช้งาน checkpoint `WAI-Nsfw-Illustrious-15.safetensors` พร้อม UI สร้างภาพแบบง่าย

## เริ่มใช้งาน
1. เปิดโน้ตบุ๊กใน Google Colab แล้วเลือก **Runtime → Change runtime type → T4 GPU**
2. รันแต่ละเซลล์ตามลำดับ; หลังติดตั้งแพ็กเกจ หาก Colab ขอให้รีสตาร์ต ให้รีสตาร์ตแล้วเริ่มรันจากเซลล์ตรวจ GPU
3. รันเซลล์ดาวน์โหลด (ไฟล์มีขนาดใหญ่ประมาณ 6.5 GB) แล้วโหลดโมเดล
4. เปิด UI, กรอก prompt แล้วกด **สร้างภาพ**

### หมายเหตุ
- ใช้เฉพาะภาพตัวละครผู้ใหญ่ที่สมมติขึ้นและเป็นไปตามกฎหมาย/ข้อกำหนดของบริการ ห้ามสร้างภาพผู้เยาว์หรือบุคคลจริงโดยไม่ได้รับความยินยอม
- UI ไม่เปิด public share link; ภาพและ prompt ไม่ควรมีข้อมูลส่วนตัว
- T4 แนะนำเริ่มที่ 768×768, Batch 1; ถ้า VRAM ไม่พอให้ลดความละเอียดหรือรีสตาร์ต runtime
- การดาวน์โหลดอาจต้องใช้ Civitai API token หากระบบร้องขอ ให้ใส่ token ในช่องรับข้อมูลลับที่แสดงขึ้นมา อย่าแชร์หรือบันทึก token ในโน้ตบุ๊ก

In [ ]:
# ติดตั้ง dependencies โดยไม่เขียนทับ PyTorch/CUDA ที่ Colab เตรียมไว้
!pip -q install -U diffusers transformers accelerate safetensors gradio requests tqdm

In [ ]:
import gc
import os
import random
import shutil
import getpass
import requests
import torch
from tqdm.auto import tqdm

if not torch.cuda.is_available():
    raise RuntimeError('ไม่พบ GPU: เลือก Runtime → Change runtime type → T4 GPU แล้วรันเซลล์นี้ใหม่')

print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

MODEL_VERSION_ID = '2167369'
CHECKPOINT_PATH = '/content/WAI-Nsfw-Illustrious-15.safetensors'
DOWNLOAD_URL = f'https://civitai.com/api/download/models/{MODEL_VERSION_ID}'

def download_checkpoint():
    if os.path.exists(CHECKPOINT_PATH) and os.path.getsize(CHECKPOINT_PATH) > 1_000_000_000:
        print(f'พบไฟล์ checkpoint แล้ว ({os.path.getsize(CHECKPOINT_PATH) / 1024**3:.2f} GB) — ข้ามการดาวน์โหลด')
        return

    token = None
    for attempt in range(2):
        params = {'token': token} if token else None
        response = requests.get(DOWNLOAD_URL, params=params, stream=True, allow_redirects=True, timeout=(30, 180))
        if response.status_code in (401, 403) and not token and attempt == 0:
            response.close()
            token = getpass.getpass('Civitai ต้องใช้ API token (รับข้อมูลแบบซ่อน): ')
            if not token.strip():
                raise RuntimeError('ไม่ได้ระบุ token; ตรวจสิทธิ์ดาวน์โหลดของโมเดลแล้วลองใหม่')
            continue
        response.raise_for_status()
        total = int(response.headers.get('content-length', 0))
        try:
            with open(CHECKPOINT_PATH, 'wb') as file, tqdm(total=total or None, unit='B', unit_scale=True, desc='กำลังดาวน์โหลดโมเดล') as progress:
                for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                    if chunk:
                        file.write(chunk)
                        progress.update(len(chunk))
        except Exception:
            if os.path.exists(CHECKPOINT_PATH):
                os.remove(CHECKPOINT_PATH)
            raise
        finally:
            response.close()
        if os.path.getsize(CHECKPOINT_PATH) < 1_000_000_000:
            os.remove(CHECKPOINT_PATH)
            raise RuntimeError('ไฟล์ที่ดาวน์โหลดมีขนาดผิดปกติ ตรวจลิงก์/สิทธิ์เข้าถึงโมเดลแล้วลองใหม่')
        print(f'ดาวน์โหลดเสร็จ: {CHECKPOINT_PATH} ({os.path.getsize(CHECKPOINT_PATH) / 1024**3:.2f} GB)')
        return
    raise RuntimeError('ดาวน์โหลดไม่สำเร็จ')

download_checkpoint()

In [ ]:
from diffusers import StableDiffusionXLPipeline, DPMSolverMultistepScheduler

if not os.path.exists(CHECKPOINT_PATH):
    raise FileNotFoundError('ไม่พบ checkpoint — รันเซลล์ดาวน์โหลดก่อน')

print('กำลังโหลด WAI NSFW v15; ครั้งแรกอาจใช้เวลาสักครู่...')
pipe = StableDiffusionXLPipeline.from_single_file(
    CHECKPOINT_PATH,
    config='stabilityai/stable-diffusion-xl-base-1.0',
    torch_dtype=torch.float16,
    use_safetensors=True,
)
pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config, use_karras_sigmas=True)
pipe.enable_vae_slicing()
pipe.enable_vae_tiling()
# Offload ไป CPU เมื่อไม่ได้ใช้งานแต่ละส่วน ช่วยให้ checkpoint SDXL ใช้กับ T4 ได้สบายขึ้น
pipe.enable_model_cpu_offload()
pipe.set_progress_bar_config(disable=True)
print('พร้อมสร้างภาพ')

In [ ]:
import gradio as gr

def generate(prompt, negative_prompt, width, height, steps, cfg, seed):
    if pipe is None:
        raise gr.Error('ยังไม่ได้โหลดโมเดล ให้รันเซลล์โหลดโมเดลก่อน')
    if not prompt or not prompt.strip():
        raise gr.Error('กรุณากรอก prompt')
    seed = int(seed)
    if seed == -1:
        seed = random.randint(0, 2**31 - 1)
    elif seed < 0:
        raise gr.Error('Seed ต้องเป็น -1 (สุ่ม) หรือเลขตั้งแต่ 0 ขึ้นไป')
    try:
        generator = torch.Generator(device='cuda').manual_seed(seed)
        result = pipe(
            prompt=prompt.strip(),
            negative_prompt=negative_prompt.strip() or None,
            width=int(width),
            height=int(height),
            num_inference_steps=int(steps),
            guidance_scale=float(cfg),
            num_images_per_prompt=1,
            generator=generator,
        )
        images = result.images
        del result
        return images, f'Seed: {seed}'
    except Exception as exc:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        message = str(exc)
        if 'out of memory' in message.lower():
            message += ' — ลดขนาดเป็น 768×768 และตรวจว่า Batch เท่ากับ 1'
        raise gr.Error(f'สร้างภาพไม่สำเร็จ: {message}')

with gr.Blocks(theme=gr.themes.Soft(), title='WAI NSFW v15 Studio') as demo:
    gr.Markdown('# WAI NSFW v15 Studio ⚡\nเริ่มที่ 768×768, Steps 24, ภาพละ 1 รูป')
    with gr.Row():
        with gr.Column():
            prompt = gr.Textbox(label='Prompt', lines=5, placeholder='บรรยายภาพที่ต้องการ')
            negative = gr.Textbox(label='Negative prompt (ไม่บังคับ)', lines=2)
            with gr.Row():
                width = gr.Dropdown([640, 768, 832, 896, 1024], value=768, label='กว้าง')
                height = gr.Dropdown([640, 768, 832, 896, 1024], value=768, label='สูง')
            steps = gr.Slider(10, 40, value=24, step=1, label='Steps')
            cfg = gr.Slider(1, 10, value=5, step=0.5, label='CFG Scale')
            seed = gr.Number(value=-1, precision=0, label='Seed (-1 = สุ่ม)')
            button = gr.Button('สร้างภาพ', variant='primary')
        with gr.Column():
            gallery = gr.Gallery(label='ภาพที่สร้าง', columns=1, height='auto')
            seed_output = gr.Textbox(label='Seed ที่ใช้', interactive=False)
    button.click(generate, inputs=[prompt, negative, width, height, steps, cfg, seed], outputs=[gallery, seed_output])

# ไม่เปิด share=True; UI ใช้งานใน Colab session นี้เท่านั้น
demo.queue().launch(share=False, inline=True)